# Generating Synthetic Golden Datasets with DeepEval

Every notebook so far used 2-3 hand-written examples — fine for learning, useless for a real eval
suite. DeepEval's **`Synthesizer`** uses an LLM to invent realistic `Golden`s for you, either from
nothing, or grounded in your own documents. This is how you get from "3 examples I typed" to "40
goldens that actually cover your app's behaviour" in a few minutes.

## 1. Install packages

We only need three libraries: `deepeval` itself, `google-genai` (so DeepEval's judge can talk to
Gemini), and `groq` (the model we're evaluating). No OpenAI key anywhere in this notebook.

In [ ]:
%pip install -q deepeval google-genai groq

## 2. Add your API keys

Two roles, kept separate the whole way through this course:

- **Groq** — the *model under test*. It free-tier serves `llama-3.3-70b-versatile`. Get a key at
  https://console.groq.com/keys
- **Gemini** — the *judge*. Every DeepEval metric needs an LLM to grade with, and we use Gemini so
  you never need an OpenAI key. Get a free key at https://aistudio.google.com/apikey

Keys are entered with `getpass` so they never get printed or saved into the notebook file.

In [ ]:
import os, getpass

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")  # skip DeepEval's anonymous telemetry

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]  # some libs read this name instead

print("Keys set.")

## 3. The generator model: `GeminiModel`

Every DeepEval metric defaults to OpenAI if you don't tell it otherwise — even a metric that
scores deterministically will still try to build an OpenAI client unless you pass `model=`. We
build **one Gemini judge** here and pass it to every metric in this notebook.

The `Synthesizer` uses the same kind of model as a metric judge — here, the same Gemini model.

In [ ]:
from deepeval.models import GeminiModel

judge = GeminiModel(model="gemini-2.5-flash-lite", api_key=os.environ["GEMINI_API_KEY"], temperature=0)
print("Judge ready:", judge.get_model_name())

## 4. Generate goldens from scratch (no docs needed)

Good for a quick smoke-test dataset when you don't have source material yet.

In [ ]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import StylingConfig

# DeepEval 4.x requires scenario/task/input_format whenever you generate goldens "from scratch"
# (there's no document to infer them from) -- a minimal StylingConfig satisfies that. We cover the
# full StylingConfig further down, once we have real docs to steer tone against.
scratch_styling = StylingConfig(
    scenario="Developers and learners asking an assistant to explain GenAI and agentic-AI concepts.",
    task="Answer questions about GenAI concepts accurately and concisely.",
    input_format="Short, natural-language questions a developer would actually type.",
)

synthesizer = Synthesizer(model=judge, max_concurrent=1, styling_config=scratch_styling)
scratch_goldens = synthesizer.generate_goldens_from_scratch(num_goldens=3)

for g in scratch_goldens:
    print("INPUT:", g.input)
    print("EXPECTED:", g.expected_output)
    print("-" * 80)


## 5. Generate goldens from your own docs (grounded)

This is the version you'll actually use: pass in chunks of your real documentation as `contexts`
(a list of chunk-groups) and the Synthesizer writes questions a user might realistically ask about
them, with an expected answer grounded in that exact text.

In [ ]:
DOCS = [
    "Retrieval-Augmented Generation (RAG) grounds an LLM's answer in external documents. At query "
    "time the system retrieves the most relevant chunks and passes them to the model as context.",
    "Temperature is a decoding setting that controls randomness. A temperature of 0 makes the "
    "model nearly deterministic, which is preferred for extraction, classification, and evaluation.",
    "An AI agent is an LLM given a goal, tools, and a loop: it plans, calls a tool, observes the "
    "result, and decides the next step until the task is done.",
]
contexts = [[doc] for doc in DOCS]  # each inner list is one "context" a golden can be grounded in

context_goldens = synthesizer.generate_goldens_from_contexts(
    contexts=contexts,
    include_expected_output=True,
    max_goldens_per_context=2,
)

for g in context_goldens:
    print("INPUT:", g.input)
    print("EXPECTED:", g.expected_output)
    print("CONTEXT:", g.context)
    print("-" * 80)

## 6. Steer the tone with `StylingConfig`

Without styling, generated questions can read generically. `StylingConfig` tells the Synthesizer
who's asking, what they're trying to do, and what a good question/answer should look like.

In [ ]:
from deepeval.synthesizer.config import StylingConfig

styling = StylingConfig(
    scenario="Developers and learners asking an assistant to explain GenAI and agentic-AI concepts.",
    task="Answer questions about GenAI concepts accurately and concisely, grounded in the docs.",
    input_format="Short, natural-language questions a developer would actually type.",
    expected_output_format="A concise, factual 1-2 sentence answer grounded in the provided context.",
)

styled_synthesizer = Synthesizer(model=judge, max_concurrent=1, styling_config=styling)
styled_goldens = styled_synthesizer.generate_goldens_from_contexts(
    contexts=contexts, include_expected_output=True, max_goldens_per_context=1,
)

for g in styled_goldens:
    print("INPUT:", g.input)
    print("EXPECTED:", g.expected_output)
    print("-" * 80)

## 7. Save to JSON and reload as an `EvaluationDataset`

Once you have goldens, run your app on each `.input`, build `LLMTestCase`s (as in notebook 1), and
evaluate — the exact same loop, just with a dataset you didn't have to write by hand.

In [ ]:
import json

all_goldens = [{"input": g.input, "expected_output": g.expected_output, "context": g.context}
               for g in context_goldens + styled_goldens]

with open("synthetic_goldens.json", "w") as f:
    json.dump(all_goldens, f, indent=2)

from deepeval.dataset import EvaluationDataset

dataset = EvaluationDataset()
dataset.add_goldens_from_json_file(file_path="synthetic_goldens.json")
print(f"Loaded {len(dataset.goldens)} goldens back into an EvaluationDataset.")

## Takeaway

`generate_goldens_from_contexts` is the one you'll use most: point it at your real docs, and each
run gives you a fresh batch of realistic, grounded goldens — no manual writing, no bias toward the
3 examples you happened to think of. Push the dataset to Confident AI with `dataset.push(alias=...)`
if you want to version and share it with your team.

Next: [`05_synthetic_conversation_data_generation.ipynb`](05_synthetic_conversation_data_generation.ipynb)
— the same idea, for whole conversations.